<a href="https://colab.research.google.com/github/aashishneupanecs24-creator/AI-Lab/blob/main/1BF24CS004_Week5_PropositionalLogic_TruthTableEnumeration.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
import pandas as pd
import itertools

def evaluate(expr, model):
    return eval(expr, {}, model)

def generate_simple_truth_table(kb, alpha):
    symbols = ['p', 'q']
    rows = []

    for truth_values in itertools.product([True, False], repeat=2):
        model = dict(zip(symbols, truth_values))

        # Evaluate each clause in the KB
        kb_evals = [evaluate(clause, model) for clause in kb]
        kb_true = all(kb_evals)

        # Evaluate the query alpha
        alpha_true = evaluate(alpha, model)

        # Entailment condition: KB => alpha (which is equivalent to: not KB or alpha)
        entails = (not kb_true) or alpha_true

        row = {
            'p': model['p'],
            'q': model['q'],
            'KB': kb_true,
            'alpha (Query)': alpha_true,
            'KB => alpha': entails
        }
        rows.append(row)

    df = pd.DataFrame(rows)
    overall_entails = df[df['KB'] == True]['alpha (Query)'].all()
    return df, overall_entails

# Scenario 1: Entailment Example
# KB: p and q are both True. Query: p
print("=== Scenario 1 (Entails Case) ===")
kb_1 = ["p and q"]
query_1 = "p"
df_1, result_1 = generate_simple_truth_table(kb_1, query_1)
display(df_1)
print(f"Does KB {kb_1} entail '{query_1}'? {'YES' if result_1 else 'NO'}\n")

# Scenario 2: Does Not Entail Example
# KB: p or q is True. Query: p
print("=== Scenario 2 (Does Not Entail Case) ===")
kb_2 = ["p or q"]
query_2 = "p"
df_2, result_2 = generate_simple_truth_table(kb_2, query_2)
display(df_2)
print(f"Does KB {kb_2} entail '{query_2}'? {'YES' if result_2 else 'NO'}")

=== Scenario 1 (Entails Case) ===


,p,q,KB,alpha (Query),KB => alpha
0,True,True,True,True,True
1,True,False,False,True,True
2,False,True,False,False,True
3,False,False,False,False,True


Does KB ['p and q'] entail 'p'? YES

=== Scenario 2 (Does Not Entail Case) ===


,p,q,KB,alpha (Query),KB => alpha
0,True,True,True,True,True
1,True,False,True,True,True
2,False,True,True,False,False
3,False,False,False,False,True


Does KB ['p or q'] entail 'p'? NO


In [7]:
import pandas as pd
import itertools
import re

def evaluate_expression(expr, model):
    """Evaluates a propositional expression safely using a truth model."""
    # Replace common logic words with python equivalents if not already done
    safe_expr = expr.lower().replace("and", " and ").replace("or", " or ").replace("not", " not ")
    return bool(eval(safe_expr, {}, model))

def analyze_entailment(kb_inputs, query_input):
    # Automatically find all unique variable names (alphabetic words except logic keywords)
    all_text = " ".join(kb_inputs + [query_input]).lower()
    words = re.findall(r'\b[a-zA-Z_][a-zA-Z0-9_]*\b', all_text)
    symbols = sorted(list({w for w in words if w not in ['and', 'or', 'not', 'true', 'false']}))

    print(f"Detected proposition variables: {symbols}")

    rows = []
    # Generate all truth assignments for the dynamic number of variables
    for truth_values in itertools.product([True, False], repeat=len(symbols)):
        model = dict(zip(symbols, truth_values))

        # Evaluate each KB clause
        kb_evals = [evaluate_expression(clause, model) for clause in kb_inputs]
        kb_true = all(kb_evals)

        # Evaluate query
        query_true = evaluate_expression(query_input, model)

        # KB => Query
        entails = (not kb_true) or query_true

        # Populate table row
        row = {sym: model[sym] for sym in symbols}
        for idx, clause in enumerate(kb_inputs):
            row[f"KB Clause {idx+1}: {clause}"] = kb_evals[idx]
        row["KB (All True)"] = kb_true
        row[f"Query: {query_input}"] = query_true
        row["Entailment Holds?"] = entails

        rows.append(row)

    df = pd.DataFrame(rows)
    overall_entails = df[df["KB (All True)"] == True][f"Query: {query_input}"].all() if any(df["KB (All True)"]) else True

    display(df)
    print(f"RESULT: KB entails '{query_input}'? {'YES (Entailed)' if overall_entails else 'NO (Not Entailed)'}")

# --- Dynamic User Input Area ---
# You can change these lists to any variable names (p, q, r, s, etc.)
user_kb = ["p and q", "q or r"]
user_query = "p"

analyze_entailment(user_kb, user_query)

Detected proposition variables: ['p', 'q', 'r']


,p,q,r,KB Clause 1: p and q,KB Clause 2: q or r,KB (All True),Query: p,Entailment Holds?
0,True,True,True,True,True,True,True,True
1,True,True,False,True,True,True,True,True
2,True,False,True,False,True,False,True,True
3,True,False,False,False,False,False,True,True
4,False,True,True,False,True,False,False,True
5,False,True,False,False,True,False,False,True
6,False,False,True,False,True,False,False,True
7,False,False,False,False,False,False,False,True


RESULT: KB entails 'p'? YES (Entailed)
